# Cuestionario 1 — confusión y ajuste por $X$

El archivo contiene 350 individuos, con tratamiento observado $T$, covariable pre-tratamiento $X$ y resultado $Y$. Dado que $X$ afecta la asignación de $T$ y también a $Y$, funciona como confusor. Por eso se comparan una estimación ingenua y una ajustada por $X$.

Las respuestas se reportan con cuatro decimales, aunque los cálculos conservan toda la precisión disponible.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path('calculos_causal_data.csv')
df = pd.read_csv(DATA_PATH)

assert df.shape == (350, 3)
assert set(df.columns) == {'T', 'X', 'Y'}
assert set(df['T'].unique()) == {0, 1}

df.head()

,T,X,Y
0,1,46.1,10.71
1,0,38.7,6.14
2,0,35.6,1.56
3,0,40.5,2.95
4,0,51.0,10.20


## 1. Diferencia de medias ingenua

Se calcula la diferencia observada entre los promedios de $Y$ de tratados y no tratados:

$$\widehat{\Delta} = \bar{Y}_{T=1} - \bar{Y}_{T=0}.$$

In [2]:
mean_treated = df.loc[df['T'] == 1, 'Y'].mean()
mean_control = df.loc[df['T'] == 0, 'Y'].mean()
naive_difference = mean_treated - mean_control

print(f'Media de Y entre tratados (T=1):     {mean_treated:.4f}')
print(f'Media de Y entre no tratados (T=0):  {mean_control:.4f}')
print(f'ANSWER1 — Diferencia ingenua:        {naive_difference:.4f}')

Media de Y entre tratados (T=1):     12.6872
Media de Y entre no tratados (T=0):  7.6712
ANSWER1 — Diferencia ingenua:        5.0160


## 2. Regresión lineal ajustada por $X$

Se estima el modelo lineal

$$Y_i = \beta_0 + \beta_T T_i + \beta_X X_i + \varepsilon_i.$$

El cálculo por mínimos cuadrados ordinarios con `numpy.linalg.lstsq` es equivalente al coeficiente de `T` que entrega `lm(Y ~ T + X)` en R.

In [3]:
design_matrix = np.column_stack([
    np.ones(len(df)),
    df['T'].to_numpy(),
    df['X'].to_numpy(),
])

intercept, beta_treatment, beta_x = np.linalg.lstsq(
    design_matrix, df['Y'].to_numpy(), rcond=None
)[0]

coefficients = pd.Series(
    {'Intercepto': intercept, 'T': beta_treatment, 'X': beta_x}
).rename('coeficiente')
display(coefficients.to_frame())
print(f'ANSWER2 — Coeficiente ajustado de T: {beta_treatment:.4f}')

,coeficiente
Intercepto,-3.828251
T,2.090679
X,0.236788


ANSWER2 — Coeficiente ajustado de T: 2.0907


## 3. Sesgo de no controlar por $X$

La diferencia solicitada es

$$\widehat{\Delta} - \widehat{\beta}_T.$$

Un valor positivo indica que la comparación ingenua sobreestima el efecto estimado luego de controlar por el confusor.

In [4]:
confounding_bias = naive_difference - beta_treatment

print(f'ANSWER3 — Delta_hat - beta_T_hat: {confounding_bias:.4f}')
print('Conclusión: la estimación ingenua sobreestima el efecto ajustado por X.')

ANSWER3 — Delta_hat - beta_T_hat: 2.9254
Conclusión: la estimación ingenua sobreestima el efecto ajustado por X.


## Respuestas

- **ANSWER1:** $\widehat{\Delta} = 5.0160$.
- **ANSWER2:** $\widehat{\beta}_T = 2.0907$.
- **ANSWER3:** $\widehat{\Delta} - \widehat{\beta}_T = 2.9254$.

Al ignorar $X$, la comparación de medias **sobreestima** el efecto ajustado en aproximadamente 2.9254 unidades. Esta interpretación causal del ajuste requiere los supuestos habituales: consistencia, positividad y ausencia de confusión no medida condicional en $X$.